# Phase 6 — endgame-heavy SFT

One adapter, one question:

> Does giving each answer **several distinct constraint paths** to itself fix
> the endgame failure that Phase 5 identified?

## What Phase 5 established

| | |
|---|---|
| best constrained SFT (`tree_salet`) | **5.6870** mean, 56.1% fail |
| classical `random` | 4.0203 |
| classical `entropy` | 3.4431 |
| base Qwen (constrained) | 7.0000, 100% fail |
| k=1 top-1 (state determines the answer) | **20.0%** |
| k=1 median rank of the answer | 7.5 of 12,972 |
| policy transfer at turn 2 | 90% agreement with its own expert |

Verdict **D**: removing invalid words entirely bought 0.49 guesses and the model
still lost to random elimination. The policy transferred; the finish did not.

## The actual defect in the data

Not a shortage of endgame states — a shortage of *paths per word*:

```
tree_salet natural:  1,612 k=1 rows / 1,612 distinct words = 1.00 paths per word
```

Every winning move is demonstrated exactly once, by one memorised route. That
teaches a 1,612-way lookup, not the procedure "read the constraints, produce the
word that satisfies them" — which is what evaluation demands on words whose
single route was never shown.

## The intervention

`build_endgame_dataset.py` rolls out **deliberately imperfect** games (SALET
opener, then randomised continuations) and labels the states with the
`tree_salet` expert — DAgger-style noise injection.

```
endgame synthetic:  6,148 k=1 rows / 2,068 words = 2.97 paths per word
MIXED:              7,760 k=1 rows / 2,068 words = 3.75 paths per word
```

Total training rows go 7,067 → 19,212, and the turn-1 constant drops from 29.3%
of the data to 10.8%.

The model still never sees a candidate list, a candidate count, or the answer.
Held-out answers are never used to generate a state. `verify_endgame_dataset.py`
asserts all of it.

## The hypothesis, and how it can fail

**Hypothesis:** many paths per word teaches constraint-following as a
*procedure*, so it generalises to held-out words; the constrained decoder then
supplies the lexicon.

**Failure mode to watch:** it may simply memorise the training answers harder.
Phase 5 already measured 33.3% top-1 on *seen* words vs 15.3% on unseen. If
Phase 6 moves seen sharply and unseen barely, the intervention did not teach a
procedure — it bought more memorisation, and the approach is wrong rather than
under-trained. The terminal probe reports that split directly and it is the
number to read first.

---

## How to run

**STEP 1** — Accelerator: **GPU T4 x2**.

**STEP 2** — Add Input, both datasets:
* the SFT package **rebuilt after Phase 6** (must contain
  `sft_package/data/tree_salet_endgame/train.jsonl`)
* the adapters dataset holding the Phase 5 `tree_salet` adapter

**STEP 3** — Set `PREV_RUN_DIR` in the config cell to the adapters path.

**STEP 4** — Run All. Budget **~3.5 h**:

| Stage | Time |
|---|---|
| train `tree_salet_endgame` (19,212 rows, 2 epochs) | ~70 min |
| 2x2 constrained evaluation | ~60 min |
| classical baselines | ~10 min |
| terminal probe, 2 models, capped | ~50 min |

**STEP 5** — Download `wordle_endgame_results.zip` (small — no weights) **and**
snapshot `/kaggle/working/wordle_endgame` as a Dataset if you want to keep the
adapter.

---
# 1. Environment

In [ ]:
import os, sys, json, time, math, random, subprocess, platform, shutil, hashlib
import importlib
from collections import Counter

def _pip(pkg):
    print(f"installing {pkg} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    import torch
except ImportError:
    _pip("torch"); import torch
for mod, pkg in [("transformers", "transformers>=4.44"),
                 ("peft", "peft>=0.11"),
                 ("accelerate", "accelerate>=0.30")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        _pip(pkg)

import torch, transformers, peft, accelerate
import numpy as np


def fix_torchao_peft_conflict():
    """peft.import_utils.is_torchao_available() RAISES on an outdated torchao
    instead of returning False, which kills get_peft_model. Kaggle ships
    torchao 0.10 while peft wants >= 0.16. We never use torchao."""
    try:
        import peft.import_utils as piu
    except Exception as e:
        return f"peft.import_utils unavailable ({e})"
    try:
        piu.is_torchao_available(); return "no conflict"
    except ImportError:
        pass
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                    "torchao"], check=False)
    importlib.invalidate_caches()
    try:
        piu.is_torchao_available(); return "resolved: uninstalled torchao"
    except ImportError:
        pass
    piu.is_torchao_available = lambda *a, **k: False
    try:
        import peft.tuners.lora.torchao as plt_ao
        plt_ao.is_torchao_available = lambda *a, **k: False
    except Exception:
        pass
    return "resolved: patched is_torchao_available -> False"


_TORCHAO_FIX = fix_torchao_peft_conflict()

print("=" * 66); print("ENVIRONMENT"); print("=" * 66)
for k, v in [("python", platform.python_version()), ("torch", torch.__version__),
             ("transformers", transformers.__version__), ("peft", peft.__version__),
             ("accelerate", accelerate.__version__), ("numpy", np.__version__),
             ("torchao fix", _TORCHAO_FIX)]:
    print(f"{k:<14} {v}")
print(f"\nCUDA available    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU               {p.name}")
    print(f"VRAM              {p.total_memory/2**30:.1f} GiB")
    print(f"bf16 supported    {torch.cuda.is_bf16_supported()}  (we use fp16)")
else:
    print("\n!! NO GPU !! Session options -> Accelerator -> GPU T4 x2, then re-run.")

---
# 2. Configuration

Everything except the dataset is identical to Phase 4/5, so the comparison
isolates endgame coverage.

In [ ]:
# ============================ EDIT THIS =====================================
DATASET_DIR  = None    # auto-detect; must contain data/tree_salet_endgame/
PREV_RUN_DIR = None    # e.g. "/kaggle/input/datasets/<user>/wordle-sft-adapters/kaggle_adapters_upload"
# ============================================================================

RUN_TRAINING   = True
RUN_EVALUATION = True
RUN_BASELINES  = True
RUN_TERMINAL_PROBE = True

# ---- the mix ---------------------------------------------------------------
USE_NATURAL = True     # sft_package/data/tree_salet/train.jsonl        (7,067)
USE_ENDGAME = True     # sft_package/data/tree_salet_endgame/train.jsonl (12,145)
ENDGAME_REPEAT = 1     # oversample the synthetic half (1 = plain mix)

# ---- model / LoRA / optimisation: IDENTICAL to Phase 4 ----------------------
MODEL_NAME   = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_NAME = "tree_salet_endgame"
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"]
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 2
PER_DEVICE_BS = 4
GRAD_ACCUM    = 4
MAX_SEQ_LEN   = 640
WARMUP_RATIO  = 0.03
WEIGHT_DECAY  = 0.0
LR_SCHEDULER  = "cosine"
FP16          = True
GRAD_CHECKPOINT = True
LOGGING_STEPS = 25
SAVE_STEPS    = 400
SAVE_TOTAL_LIMIT = 1
SEED          = 20260817

# ---- evaluation ------------------------------------------------------------
MAX_GUESSES        = 6
GEN_MAX_NEW_TOKENS = 8
EVAL_BATCH         = 16
SHOW_CANDIDATE_COUNT = False        # never, in training or evaluation

CONSTRAINED_CHUNK = 512
CONSTRAINED_PRUNE = True
LENGTH_NORMALISE  = False

# The 2x2. Phase 5's tree_salet number (5.6870) was measured with ban=False, so
# the no-ban cells are what make the comparison like-for-like; the ban cells
# measure the decoder change separately.
EVAL_MATRIX = [
    ("tree_salet_endgame", True),
    ("tree_salet_endgame", False),
    ("tree_salet",         True),
    ("tree_salet",         False),   # reproduces Phase 5: expect 5.6870
]

TERMINAL_MAX_STATES = {1: 80, 2: 30, 3: 27}
TERMINAL_MODELS = ["tree_salet_endgame", "tree_salet"]
PROBE_LOG_EVERY = 10

# ---- reference numbers from Phase 5, for the comparison table --------------
PHASE5 = {
    "tree_salet_constrained_noban": 5.6870,
    "tree_salet_constrained_noban_fail": 56.10,
    "tree_salet_unconstrained": 6.1748,
    "base_qwen_constrained": 7.0000,
    "classical_entropy": 3.4431,
    "classical_random": 4.0203,
    "k1_top1": 20.0,
    "k1_seen": 33.33,
    "k1_unseen": 15.25,
    "k1_median_rank": 7.5,
}

# ---- paths -----------------------------------------------------------------
WORK_DIR     = "/kaggle/working/wordle_endgame"
RESULTS_ROOT = "/kaggle/working/results_endgame"
RESULTS_ZIP  = "/kaggle/working/wordle_endgame_results.zip"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

def set_seed_everywhere(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)

set_seed_everywhere()
print(f"adapter        {ADAPTER_NAME}")
print(f"mix            natural={USE_NATURAL} endgame={USE_ENDGAME} "
      f"(endgame x{ENDGAME_REPEAT})")
_ngpu = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"effective batch {PER_DEVICE_BS} x {GRAD_ACCUM} x {_ngpu} GPU(s) = "
      f"{PER_DEVICE_BS*GRAD_ACCUM*_ngpu}")
if _ngpu > 1:
    print(f"  NOTE: Trainer uses all {_ngpu} GPUs, so the real effective batch "
          f"is {PER_DEVICE_BS*GRAD_ACCUM*_ngpu}. Phase 4 ran the same way "
          f"(7,173 rows -> 448 steps confirms it), so this stays comparable.")
print(f"expected steps ~{2*len(range(19212))//(PER_DEVICE_BS*GRAD_ACCUM*_ngpu)} "
      f"at ~3.3 s/step -> see section 6 for the real count")
print(f"eval matrix    {EVAL_MATRIX}")
print(f"work dir       {WORK_DIR}")

---
# 3. Locate and validate the dataset

Fails loudly, and specifically names the endgame file if it is missing — that is
the one new requirement versus Phase 5.

In [ ]:
import glob

REQUIRED_FILES = [
    "sft_package/data/tree_salet/train.jsonl",
    "sft_package/data/tree_salet_endgame/train.jsonl",   # NEW in Phase 6
    "sft_package/data/disagreement/contrast.jsonl",
    "sft_package/eval/val_answers.jsonl",
    "sft_package/eval/train_answers.jsonl",
    "code/wordle_solver.py",
    "code/tree_search.py",
    "code/generate_trajectories.py",
    "artifacts/answers.txt",
    "artifacts/valid_guesses.txt",
    "artifacts/feedback_matrix.npy",
]

def _has_all(d):
    try:
        return all(os.path.exists(os.path.join(d, f)) for f in REQUIRED_FILES)
    except OSError:
        return False

def _search(root, max_depth=7):
    if not os.path.isdir(root):
        return None
    for depth in range(0, 6):
        pat = os.path.join(root, *(["*"] * depth)) if depth else root
        for d in sorted(glob.glob(pat)):
            if os.path.isdir(d) and _has_all(d):
                return d
    base = os.path.abspath(root).rstrip(os.sep).count(os.sep)
    best = None
    for dirpath, dirnames, _ in os.walk(root):
        if os.path.abspath(dirpath).rstrip(os.sep).count(os.sep) - base > max_depth:
            dirnames[:] = []; continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if _has_all(dirpath):
            if best is None or len(dirpath) < len(best):
                best = dirpath
            dirnames[:] = []
    return best

def locate_dataset(explicit=None):
    if explicit:
        for c in (explicit, os.path.join(explicit, "kaggle_upload")):
            if _has_all(c):
                return c
        hit = _search(explicit)
        if hit:
            return hit
    for root in ("/kaggle/input", ".", "/kaggle/working"):
        hit = _search(root)
        if hit:
            return hit
    msg = ["Could not find the dataset package.", ""]
    eg = "sft_package/data/tree_salet_endgame/train.jsonl"
    near = []
    for dirpath, dirnames, _ in os.walk("/kaggle/input"):
        if os.path.exists(os.path.join(dirpath, "sft_package/data/tree_salet/train.jsonl")):
            near.append(dirpath)
    if near:
        msg += ["Found a package WITHOUT the Phase 6 endgame file:"] + \
               [f"    {n}" for n in near] + \
               ["", f"Missing: {eg}", "",
                "Rebuild it locally and re-upload:",
                "    python build_endgame_dataset.py",
                "    python verify_endgame_dataset.py",
                "    python prepare_kaggle_dataset.py --dest uploads/kaggle_upload"]
    else:
        msg.append("Nothing that looks like the SFT package under /kaggle/input.")
    raise FileNotFoundError("\n".join(msg))

DATA_ROOT = locate_dataset(DATASET_DIR)
SFT_DIR = os.path.join(DATA_ROOT, "sft_package")
ARTIFACTS = os.path.join(DATA_ROOT, "artifacts")
CODE_DIR = os.path.join(DATA_ROOT, "code")
print(f"dataset root : {DATA_ROOT}")
for f in REQUIRED_FILES:
    p = os.path.join(DATA_ROOT, f)
    print(f"  {os.path.getsize(p)/1024:>10.1f} KiB  {f}")

def ensure_code_on_path():
    if CODE_DIR not in sys.path:
        sys.path.insert(0, CODE_DIR)

ensure_code_on_path()
from wordle_solver import (load_artifacts, SolverConfig, make_solver, play_game,
                           feedback_code, code_to_pattern, ALL_GREEN)
from generate_trajectories import derive_constraints, render_prompt

BUNDLE = load_artifacts(ARTIFACTS, mmap=True)
VOCAB = BUNDLE.vocab
LEGAL_GUESSES = set(w.upper() for w in VOCAB.guesses)
VAL_ANSWERS = [json.loads(l)["answer"].upper()
               for l in open(os.path.join(SFT_DIR, "eval/val_answers.jsonl"),
                             encoding="utf-8")]
print(f"\nlegal guesses {len(LEGAL_GUESSES)}   held-out answers {len(VAL_ANSWERS)}")
assert len(LEGAL_GUESSES) == 12972
assert len(VAL_ANSWERS) == 246

---
# 4. Model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if TOKENIZER.pad_token is None:
    TOKENIZER.pad_token = TOKENIZER.eos_token
TOKENIZER.padding_side = "right"

def load_base_model():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map=None,
        trust_remote_code=True)
    m.config.use_cache = False
    return m

print(f"model {MODEL_NAME}\nvocab {len(TOKENIZER)}  eos {TOKENIZER.eos_token_id}")

---
# 5. The mix

The whole experiment is this cell. Read the printed `paths per word` line: the
natural data is 1.00 and that is the defect being corrected.

In [ ]:
from torch.utils.data import Dataset

def load_jsonl(p):
    with open(p, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

NATURAL = load_jsonl(os.path.join(SFT_DIR, "data/tree_salet/train.jsonl"))
ENDGAME = load_jsonl(os.path.join(SFT_DIR, "data/tree_salet_endgame/train.jsonl"))

ROWS = []
if USE_NATURAL:
    ROWS += NATURAL
if USE_ENDGAME:
    ROWS += ENDGAME * ENDGAME_REPEAT
random.Random(SEED).shuffle(ROWS)

def bucket(n):
    return "1" if n == 1 else "2" if n == 2 else "3" if n == 3 \
        else "4-10" if n <= 10 else ">10"

def describe(label, rows):
    n = len(rows)
    bc = Counter(bucket(r["meta"]["n_candidates"]) for r in rows)
    tc = Counter(r["meta"]["turn"] for r in rows)
    k1 = [r for r in rows if r["meta"]["n_candidates"] == 1]
    per = Counter(r["completion"] for r in k1)
    ppw = len(k1) / max(len(per), 1)
    print(f"\n{label}  ({n} rows)")
    print(f"  candidates : {dict(sorted(bc.items()))}")
    print(f"  turns      : {dict(sorted(tc.items()))}")
    print(f"  turn-1 share: {100*tc.get(1,0)/n:.1f}%")
    print(f"  k=1        : {len(k1)} rows / {len(per)} words "
          f"= {ppw:.2f} paths per word")
    return {"n_rows": n, "buckets": dict(bc), "turns": dict(tc),
            "turn1_pct": round(100*tc.get(1,0)/n, 2),
            "k1_rows": len(k1), "k1_words": len(per),
            "k1_paths_per_word": round(ppw, 3)}

print("=" * 66); print("DATASET COMPOSITION"); print("=" * 66)
DATA_STATS = {
    "natural": describe("natural (Phase 4/5)", NATURAL),
    "endgame": describe("endgame synthetic (Phase 6)", ENDGAME),
    "mixed":   describe("MIXED -> training set", ROWS),
}
DATA_STATS["endgame_repeat"] = ENDGAME_REPEAT

print("\n" + "=" * 66)
print(f"paths per word at k=1:  natural {DATA_STATS['natural']['k1_paths_per_word']}"
      f"  ->  mixed {DATA_STATS['mixed']['k1_paths_per_word']}")
print("=" * 66)

# The prompt must be unchanged from Phase 4/5, or nothing is comparable.
assert not any("Possible answers remaining" in r["prompt"] for r in ROWS)
assert all(r["prompt"].rstrip().endswith("Next guess:") for r in ROWS[:2000])
print("\nprompt format unchanged from Phase 4/5  OK")


class WordleSFTDataset(Dataset):
    """prompt -> completion, prompt masked out of the loss."""
    def __init__(self, rows, tokenizer, max_len=MAX_SEQ_LEN):
        self.rows, self.tok, self.max_len = rows, tokenizer, max_len
        self.n_truncated = 0
        self._cache = {}
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        if i in self._cache:
            return self._cache[i]
        r = self.rows[i]
        p_ids = self.tok(r["prompt"], add_special_tokens=False)["input_ids"]
        c_ids = self.tok(" " + r["completion"],
                         add_special_tokens=False)["input_ids"]
        c_ids = c_ids + [self.tok.eos_token_id]
        keep = self.max_len - len(c_ids)
        if len(p_ids) > keep:
            p_ids = p_ids[-keep:]          # drop OLDEST prompt tokens
            self.n_truncated += 1
        item = {"input_ids": p_ids + c_ids,
                "labels": [-100] * len(p_ids) + c_ids,
                "attention_mask": [1] * (len(p_ids) + len(c_ids))}
        self._cache[i] = item
        return item

def collate(batch, pad_id):
    n = max(len(b["input_ids"]) for b in batch)
    out = {"input_ids": [], "labels": [], "attention_mask": []}
    for b in batch:
        d = n - len(b["input_ids"])
        out["input_ids"].append(b["input_ids"] + [pad_id] * d)
        out["labels"].append(b["labels"] + [-100] * d)
        out["attention_mask"].append(b["attention_mask"] + [0] * d)
    return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

_lens = []
for r in ROWS[:1500]:
    _lens.append(len(TOKENIZER(r["prompt"], add_special_tokens=False)["input_ids"])
                 + len(TOKENIZER(" " + r["completion"],
                                 add_special_tokens=False)["input_ids"]) + 1)
_lens = np.array(_lens)
print(f"token lengths: mean {_lens.mean():.0f}  p95 {np.percentile(_lens,95):.0f}"
      f"  max {_lens.max()}   MAX_SEQ_LEN={MAX_SEQ_LEN} -> "
      f"{100*(_lens>MAX_SEQ_LEN).mean():.2f}% truncated")

---
# 6. Train `tree_salet_endgame`

Identical hyperparameters to Phase 4. The dataset is ~2.7x larger, so 2 epochs
is ~2.7x the steps — that is a consequence of more data, not a changed setting.

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel
from transformers import Trainer, TrainingArguments

def make_lora_model():
    fix_torchao_peft_conflict()
    base = load_base_model()
    m = get_peft_model(base, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"))
    # amp keeps fp32 master weights and refuses to unscale fp16 grads.
    n = 0
    for _, p in m.named_parameters():
        if p.requires_grad and p.dtype == torch.float16:
            p.data = p.data.float(); n += 1
    if n:
        print(f"  cast {n} trainable tensors fp16 -> fp32 (amp requirement)")
    return m

TRAIN_CONFIG = {}
out_dir = os.path.join(WORK_DIR, ADAPTER_NAME)

if RUN_TRAINING:
    print("=" * 66); print(f"TRAINING {ADAPTER_NAME} on {len(ROWS)} rows")
    print("=" * 66)
    set_seed_everywhere()
    ds = WordleSFTDataset(ROWS, TOKENIZER)
    model = make_lora_model()
    model.print_trainable_parameters()
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=PER_DEVICE_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHEDULER,
        warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY,
        fp16=FP16, bf16=False, gradient_checkpointing=GRAD_CHECKPOINT,
        logging_steps=LOGGING_STEPS, save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT, save_strategy="steps",
        report_to=[], seed=SEED, data_seed=SEED, optim="adamw_torch",
        max_grad_norm=1.0, dataloader_num_workers=2,
        remove_unused_columns=False, disable_tqdm=False)
    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=lambda b: collate(b, TOKENIZER.pad_token_id))
    t0 = time.perf_counter()
    trainer.train()
    secs = time.perf_counter() - t0
    trainer.save_model(out_dir)
    hist = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    TRAIN_CONFIG = {
        "name": ADAPTER_NAME, "n_train_examples": len(ROWS),
        "n_truncated": ds.n_truncated, "model_name": MODEL_NAME,
        "learning_rate": LEARNING_RATE, "num_epochs": NUM_EPOCHS,
        "per_device_batch_size": PER_DEVICE_BS,
        "gradient_accumulation": GRAD_ACCUM,
        "effective_batch_size": PER_DEVICE_BS * GRAD_ACCUM,
        "max_seq_len": MAX_SEQ_LEN, "lr_scheduler": LR_SCHEDULER,
        "warmup_ratio": WARMUP_RATIO, "fp16": FP16,
        "gradient_checkpointing": GRAD_CHECKPOINT, "seed": SEED,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "training_seconds": round(secs, 1),
        "global_steps": trainer.state.global_step,
        "first_loss": hist[0] if hist else None,
        "final_loss": hist[-1] if hist else None,
        "data_stats": DATA_STATS,
    }
    json.dump(TRAIN_CONFIG, open(os.path.join(out_dir, "training_config.json"),
                                 "w", encoding="utf-8"), indent=2, default=str)
    print(f"\ntrained in {secs/60:.1f} min, {trainer.state.global_step} steps, "
          f"loss {hist[0]:.4f} -> {hist[-1]:.4f}")
    del model, trainer; torch.cuda.empty_cache()
else:
    p = os.path.join(out_dir, "training_config.json")
    if os.path.exists(p):
        TRAIN_CONFIG = json.load(open(p))
        print(f"training skipped; existing adapter at {out_dir}")
    else:
        print("training skipped and no existing adapter")

---
# 7. Constrained decoding

Unchanged from Phase 5 so the comparison holds. The module docstring below is
the full specification; the short version is that the model never emits free
text — all 12,972 legal words are scored and the argmax is played.

Phase 6 turns **repeat banning on**, so the verification cell also checks that
sequential banning walks the global ranking exactly. That check caught a real
bug in Phase 5 (the ban masked the pruning bound but not the final scores).

In [ ]:
"""
constrained_decode.py — exact argmax over a fixed legal-word set.

The question this answers is

    "Which of these 12,972 legal Wordle words should I play?"

not

    "Generate arbitrary text and see whether it happens to be a word."

Nothing is filtered after the fact. The model never emits free text at all in
constrained mode: every legal word is *scored*, and the highest-scoring one is
played.

--------------------------------------------------------------------------
Definition of the score
--------------------------------------------------------------------------
For a prompt `p` and a legal word `w`, tokenize exactly as the SFT data did --
`" " + w` -- and append EOS. Then

    score(w) = log P(w | p) = sum_i log P(t_i | p, t_0..t_{i-1})

summed over the word's tokens **and the EOS token**.

Including EOS matters. Without it, a word whose token sequence is a prefix of a
longer word's is scored on a strictly smaller set of constraints and is
systematically over-ranked; with EOS the scores are log-probabilities of
complete strings, so they are directly comparable across different token
lengths. No length normalisation is applied: `score(w)` is exactly the
probability the model assigns to playing `w`, which is the quantity we want to
argmax. (`length_normalise=True` is available for a sensitivity check but is a
heuristic, not the default.)

--------------------------------------------------------------------------
How it is computed
--------------------------------------------------------------------------
Naively this is 12,972 forward passes per turn. Instead:

1. The prompt is run **once** with `use_cache=True`, giving a KV cache and the
   next-token distribution `lp0` over the whole vocabulary.
2. `lp0` already gives the first-token log-probability of every legal word, for
   free -- no forward pass.
3. The remaining tokens are scored by teacher forcing: the prompt's KV cache is
   expanded to a batch of `chunk` rows and the padded `[chunk, L]` word-token
   matrix is pushed through in one forward pass. Causal masking makes the
   right-hand padding inert.

Exact branch-and-bound pruning (`prune=True`, on by default and *exact*):
every per-token log-probability is <= 0, so

    score(w) <= lp0[first_token(w)]

is a valid upper bound. Words are visited in descending order of that bound;
once the best fully-scored word beats the bound of every unvisited word, no
unvisited word can win and the scan stops. The returned argmax is identical to
scoring all 12,972 -- `verify_against_full()` asserts exactly that.

--------------------------------------------------------------------------
What is NOT done here
--------------------------------------------------------------------------
- The candidate-answer list is never consulted. The scorer ranks the full legal
  guess pool; it has no idea which words are still possible.
- The hidden answer is never consulted.
- Repeats are not banned by default (`banned` is opt-in), because banning them
  would be a policy change, not a vocabulary constraint.
"""

import numpy as np
import torch
import torch.nn.functional as F

NEG_INF = float("-inf")


# ---------------------------------------------------------------------------
# KV-cache compatibility
#
# transformers has moved the cache representation twice (legacy tuple ->
# DynamicCache.key_cache/value_cache -> Cache.layers). Rather than pin a
# version, read whichever layout is present and rebuild through the same one.
# `LegalWordScorer.self_test()` verifies the result numerically at runtime, so
# a layout this shim gets wrong fails loudly instead of scoring garbage.
# ---------------------------------------------------------------------------
def _cache_layers(cache):
    if hasattr(cache, "layers"):                        # transformers >= 5
        return [(lyr.keys, lyr.values) for lyr in cache.layers]
    if hasattr(cache, "key_cache"):                     # transformers 4.x
        return list(zip(cache.key_cache, cache.value_cache))
    return [(k, v) for k, v in cache]                   # legacy tuple-of-tuples


def _rebuild_cache(template, layers):
    """Rebuild a cache object of the same kind as `template` from `layers`."""
    if hasattr(template, "layers"):
        import copy
        new = copy.deepcopy(template)
        for lyr, (k, v) in zip(new.layers, layers):
            lyr.keys, lyr.values = k, v
        return new
    if hasattr(template, "key_cache"):
        from transformers.cache_utils import DynamicCache
        new = DynamicCache()
        new.key_cache = [k for k, _ in layers]
        new.value_cache = [v for _, v in layers]
        return new
    return tuple(layers)


def expand_cache(cache, n):
    """Repeat a batch-1 KV cache to batch `n` without recomputing the prompt."""
    out = []
    for k, v in _cache_layers(cache):
        assert k.shape[0] == 1, f"expected batch-1 cache, got {k.shape[0]}"
        out.append((k.expand(n, *k.shape[1:]).contiguous(),
                    v.expand(n, *v.shape[1:]).contiguous()))
    return _rebuild_cache(cache, out)


# ---------------------------------------------------------------------------
class LegalWordScorer:
    """Scores every word in a fixed legal set under a causal LM."""

    def __init__(self, tokenizer, words, device="cuda", chunk=512,
                 length_normalise=False):
        self.tok = tokenizer
        self.words = list(words)
        self.device = device
        self.chunk = chunk
        self.length_normalise = length_normalise
        self.n = len(self.words)

        eos = tokenizer.eos_token_id
        seqs = []
        for w in self.words:
            # EXACTLY the training-time tokenization: " " + WORD, then EOS.
            ids = tokenizer(" " + w, add_special_tokens=False)["input_ids"]
            seqs.append(ids + [eos])
        self.max_len = max(len(s) for s in seqs)

        pad = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else eos
        tokens = np.full((self.n, self.max_len), pad, dtype=np.int64)
        mask = np.zeros((self.n, self.max_len), dtype=np.float32)
        for i, s in enumerate(seqs):
            tokens[i, :len(s)] = s
            mask[i, :len(s)] = 1.0

        self.tokens = torch.from_numpy(tokens).to(device)      # [N, L]
        self.mask = torch.from_numpy(mask).to(device)          # [N, L]
        self.lengths = torch.from_numpy(mask.sum(1)).to(device)
        self.first_tok = self.tokens[:, 0].clone()             # [N]
        self.index = {w: i for i, w in enumerate(self.words)}

    # -- internals ----------------------------------------------------------
    @torch.no_grad()
    def _prompt_pass(self, model, prompt):
        ids = self.tok(prompt, return_tensors="pt",
                       add_special_tokens=False)["input_ids"].to(self.device)
        out = model(input_ids=ids, use_cache=True)
        lp0 = F.log_softmax(out.logits[0, -1].float(), dim=-1)   # [V]
        return out.past_key_values, lp0, ids.shape[1]

    @torch.no_grad()
    def _score_rows(self, model, cache, lp0, prompt_len, rows):
        """Exact log P(w | prompt) for the word indices in `rows`."""
        idx = rows.to(self.device)
        toks = self.tokens[idx]                                  # [C, L]
        msk = self.mask[idx]
        C, L = toks.shape

        total = lp0[toks[:, 0]] * msk[:, 0]                      # token 0, free
        if L > 1:
            big = expand_cache(cache, C)
            attn = torch.ones(C, prompt_len + L, dtype=torch.long,
                              device=self.device)
            pos = torch.arange(prompt_len, prompt_len + L,
                               device=self.device).unsqueeze(0).expand(C, L)
            out = model(input_ids=toks, past_key_values=big,
                        attention_mask=attn, position_ids=pos, use_cache=False)
            # logits[:, i] predicts token i+1, so positions 1..L-1 read 0..L-2.
            for i in range(1, L):
                lp = F.log_softmax(out.logits[:, i - 1].float(), dim=-1)
                total = total + lp.gather(1, toks[:, i:i + 1]).squeeze(1) * msk[:, i]
            del out, big
        if self.length_normalise:
            total = total / msk.sum(1)
        return total

    # -- public API ---------------------------------------------------------
    @torch.no_grad()
    def score_all(self, model, prompt):
        """Score every legal word. No pruning. Returns a [N] float tensor."""
        cache, lp0, plen = self._prompt_pass(model, prompt)
        out = torch.empty(self.n, dtype=torch.float32, device=self.device)
        for s in range(0, self.n, self.chunk):
            rows = torch.arange(s, min(s + self.chunk, self.n))
            out[rows.to(self.device)] = self._score_rows(
                model, cache, lp0, plen, rows)
        return out

    @torch.no_grad()
    def argmax(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Highest-scoring legal word.

        With `prune=True` this is the *same* word `score_all().argmax()` gives;
        the bound is exact, not a heuristic. Returns
        `(word, score, n_chunks_scored, margin)`.

        `margin` is the gap to the runner-up **among words actually scored**.
        With pruning on, unvisited words are known to score below the winner but
        could sit above the runner-up, so `margin` is an upper bound on the true
        margin. It is a diagnostic, never an input to a decision.
        """
        cache, lp0, plen = self._prompt_pass(model, prompt)

        bound = lp0[self.first_tok].clone()                      # [N] upper bd
        # `ban_mask` marks words that must NOT be returned, for any reason.
        # It is applied to the bound (so they sort last and prune early) AND to
        # the scores (so one cannot win from the tail of a live chunk). Masking
        # only the bound is a real bug we shipped once: the excluded word still
        # received a genuine score and could come out on top.
        ban_mask = torch.zeros(self.n, dtype=torch.bool, device=self.device)

        if allowed_idx is not None:
            keep = torch.zeros(self.n, dtype=torch.bool, device=self.device)
            keep[torch.as_tensor(np.asarray(allowed_idx), device=self.device)] = True
            ban_mask |= ~keep
            bound = bound.masked_fill(~keep, NEG_INF)

        if banned:
            hit = [self.index[b] for b in banned if b in self.index]
            if hit:
                ix = torch.tensor(hit, device=self.device)
                bound[ix] = NEG_INF     # sorts them last
                ban_mask[ix] = True     # and removes them from the scores

        # Only ever visit words that could win. Sorting the whole vocabulary and
        # relying on the -inf bound to skip the rest still pushes a full chunk
        # of masked-out words through the model: a 3-word admissible set cost
        # MORE than an unfiltered decision (measured 87s vs 32s on CPU) because
        # both scored 512 rows. Restricting the scan pool to the admissible set
        # makes a small set genuinely cheap, which is the common case once the
        # feedback filter bites.
        if allowed_idx is not None:
            pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
            pool = pool[~ban_mask[pool]]
            if pool.numel() == 0:                    # everything banned
                pool = torch.as_tensor(np.asarray(allowed_idx), device=self.device)
        else:
            pool = torch.arange(self.n, device=self.device)
        order = pool[torch.argsort(bound[pool], descending=True)]
        n_scan = int(order.numel())

        best_i, best_s, second = -1, NEG_INF, NEG_INF
        n_chunks = 0
        for s in range(0, n_scan, self.chunk):
            rows = order[s:s + self.chunk]
            if bound[rows[0]] == NEG_INF:
                break                                    # all remaining banned
            if best_s >= bound[rows[0]].item():
                break                # no unvisited word can beat the incumbent
            sc = self._score_rows(model, cache, lp0, plen, rows.cpu())
            # A banned word can still land in the tail of an otherwise-live
            # chunk. Masking the bound alone is not enough -- the score has to
            # be masked too, or the ban is silently ignored.
            sc = sc.masked_fill(ban_mask[rows], NEG_INF)
            n_chunks += 1
            top2 = torch.topk(sc, min(2, sc.numel()))
            if top2.values[0].item() > best_s:
                second = max(second, best_s)
                if top2.values.numel() > 1:
                    second = max(second, top2.values[1].item())
                best_s = top2.values[0].item()
                best_i = rows[top2.indices[0]].item()
            elif top2.values[0].item() > second:
                second = top2.values[0].item()

        margin = None if second == NEG_INF else best_s - second
        return self.words[best_i], best_s, n_chunks, margin

    @torch.no_grad()
    def rank_of(self, scores, words):
        """1-based ranks of `words` under a full `score_all` vector."""
        order = torch.argsort(scores, descending=True)
        pos = torch.empty_like(order)
        pos[order] = torch.arange(order.numel(), device=order.device)
        return {w: int(pos[self.index[w]].item()) + 1
                for w in words if w in self.index}

    # -- feedback-consistent selection (Phase 7) ----------------------------
    @torch.no_grad()
    def select(self, model, prompt, banned=None, allowed_idx=None, prune=True):
        """Pick a word, optionally restricted to an admissible subset.

        `allowed_idx` is an index array into `self.words`. It is intended to
        carry the FEEDBACK-CONSISTENT set: the legal words that would have
        produced exactly the feedback already observed. That set is a pure
        function of the prompt (history + the public word list) -- it never
        touches the answer list -- so restricting to it is the same class of
        move as restricting to legal words.

        Returns a dict, not a tuple, so callers can record why a word was
        chosen. The distinction matters for interpreting results:

            forced       len(allowed) == 1. The filter determined the word.
                         The model contributed nothing and this must NOT be
                         counted as a model decision.
            model_chosen len(allowed) > 1. The model ranked the admissible
                         words and picked one.

        Reporting these together would let a decoder improvement masquerade as
        a model improvement.
        """
        n_allowed = self.n if allowed_idx is None else int(len(allowed_idx))

        if allowed_idx is not None and n_allowed == 0:
            # Cannot happen while the answer is legal and the feedback honest,
            # but degrade to the full pool rather than crash a 4-hour run.
            allowed_idx, n_allowed = None, self.n

        if allowed_idx is not None and n_allowed == 1:
            i = int(allowed_idx[0])
            return {"word": self.words[i], "score": None, "n_chunks": 0,
                    "margin": None, "n_allowed": 1, "forced": True,
                    "model_chosen": False}

        w, s, nch, margin = self.argmax(model, prompt, banned=banned,
                                        allowed_idx=allowed_idx, prune=prune)
        return {"word": w, "score": s, "n_chunks": nch, "margin": margin,
                "n_allowed": n_allowed, "forced": False, "model_chosen": True}

    # -- correctness --------------------------------------------------------
    @torch.no_grad()
    def self_test(self, model, prompt, n_probe=32, atol=None):
        """Check cache-reuse scoring against naive full-sequence scoring.

        This is the guard on the KV-cache shim above. A shim that mishandles a
        transformers version produces essentially random scores -- wrong by
        many nats, with the ordering destroyed. That is the failure this must
        catch, and it is enormous.

        What it must NOT flag is float16 rounding. The two paths run different
        matmul shapes (one sequence of length P+L, versus a batch of L-token
        rows against an expanded P-token cache), so fp16 reduction order
        differs and summed log-probs disagree at the 0.01-0.1 nat level. That
        is arithmetic noise, not a broken cache.

        So the test is two-sided:
          * absolute deviation under a dtype-aware tolerance, and
          * the two score vectors still rank the probe words the same way
            (correlation ~1). A broken shim cannot preserve the ranking.

        Note this discrepancy is a *validation* artifact only. Every one of the
        12,972 words is scored through the same fast path, so the ranking the
        argmax is read off is internally consistent.

        Probe indices are spread evenly across the whole word list, not taken
        from the front: if the prompt cache were mutated in place by the first
        chunk's forward pass, only words in *later* chunks would be wrong, and
        a probe drawn from the front would miss it entirely.
        """
        dtype = next(model.parameters()).dtype
        if atol is None:
            atol = 0.02 if dtype in (torch.float32, torch.float64) else 0.40

        fast = self.score_all(model, prompt)
        p_ids = self.tok(prompt, return_tensors="pt",
                         add_special_tokens=False)["input_ids"].to(self.device)
        n_probe = min(n_probe, self.n)
        probe = [int(round(i * (self.n - 1) / max(n_probe - 1, 1)))
                 for i in range(n_probe)]

        got, ref_all = [], []
        for i in probe:
            ids = self.tokens[i][self.mask[i] > 0].unsqueeze(0)
            full = torch.cat([p_ids, ids], dim=1)
            logits = model(input_ids=full, use_cache=False).logits[0].float()
            lp = F.log_softmax(logits, dim=-1)
            start = p_ids.shape[1] - 1
            ref = sum(lp[start + j, ids[0, j]].item() for j in range(ids.shape[1]))
            if self.length_normalise:
                ref /= ids.shape[1]
            ref_all.append(ref)
            got.append(fast[i].item())

        a = np.asarray(got, dtype=np.float64)
        b = np.asarray(ref_all, dtype=np.float64)
        worst = float(np.max(np.abs(a - b)))
        spread = float(b.max() - b.min())
        corr = (float(np.corrcoef(a, b)[0, 1])
                if a.std() > 1e-9 and b.std() > 1e-9 else 1.0)

        assert corr > 0.999, (
            f"constrained scorer does not preserve the ranking of the naive "
            f"scorer (corr={corr:.6f}). The KV-cache shim in expand_cache() "
            f"does not match this transformers version -- do not trust "
            f"constrained results.")
        assert worst < atol, (
            f"constrained scorer disagrees with naive scoring by {worst:.4f} "
            f"nats (tol {atol} for dtype {dtype}), across a probe score spread "
            f"of {spread:.1f} nats. Ranking is preserved (corr={corr:.6f}), so "
            f"this looks like arithmetic noise rather than a broken cache -- "
            f"but it is larger than expected. Investigate before trusting the "
            f"numbers.")
        return {"max_abs_dev": worst, "corr": corr, "spread": spread,
                "atol": atol, "dtype": str(dtype), "n_probe": n_probe}

    @torch.no_grad()
    def verify_against_full(self, model, prompt):
        """Assert the pruned argmax equals the unpruned argmax."""
        full = self.score_all(model, prompt)
        w_full = self.words[int(full.argmax().item())]
        w_prune, _, n_chunks, _ = self.argmax(model, prompt, prune=True)
        assert w_full == w_prune, (
            f"pruning changed the answer: full={w_full} pruned={w_prune}. "
            f"The branch-and-bound bound is wrong.")
        return w_full, n_chunks


# ---------------------------------------------------------------------------
class HardModeFilter:
    """The legal words consistent with every piece of feedback received.

    A word `w` is admissible iff, for every past guess `g` with observed
    pattern `p`, `feedback_code(g, w) == p`. That is precisely the set a
    hard-mode Wordle player can compute from their own board.

    WHAT THIS IS NOT: the candidate set. Two sets are easy to conflate and the
    difference is the whole justification --

        candidate set   answers consistent with feedback   pool = 2,315 answers
                        -> uses the ANSWER LIST, privileged, never used here
        hard-mode set   legal guesses consistent with it   pool = 12,972 legal
                        -> a pure function of the prompt + the public word list

    The model is never shown this set, its size, the answer, or the answer
    list. It is a decoder-side restriction on which words may be selected,
    exactly like the legal-word constraint.

    Refinement is incremental, so cost is dominated by the first turn and
    collapses immediately after (measured: 12,972 -> ~211 -> ~7 -> ~2).
    """

    def __init__(self, legal_words_lower, feedback_code_fn):
        self.words = list(legal_words_lower)
        self._fb = feedback_code_fn
        self.history = []

    def refine(self, guess, code):
        """Apply one (guess, feedback) pair. `guess` lower-case, `code` int."""
        g = guess.lower()
        self.words = [w for w in self.words if self._fb(g, w) == code]
        self.history.append((g, code))
        return self

    def indices(self, scorer):
        """Index array into `scorer.words` (which are upper-case)."""
        return np.array([scorer.index[w.upper()] for w in self.words
                         if w.upper() in scorer.index], dtype=np.int64)

    def contains(self, word):
        return word.lower() in self.words

    def __len__(self):
        return len(self.words)


# ---------------------------------------------------------------------------
LEGAL_WORDS_SORTED = sorted(LEGAL_GUESSES)
assert len(LEGAL_WORDS_SORTED) == 12972

SCORER = None
_SCORER_VERIFIED = False

def build_scorer(device="cuda"):
    global SCORER
    if SCORER is None:
        t0 = time.perf_counter()
        SCORER = LegalWordScorer(TOKENIZER, LEGAL_WORDS_SORTED, device=device,
                                 chunk=CONSTRAINED_CHUNK,
                                 length_normalise=LENGTH_NORMALISE)
        print(f"scorer over {SCORER.n} words (max {SCORER.max_len} tokens) "
              f"in {time.perf_counter()-t0:.1f}s")
    return SCORER

def verify_scorer(model):
    """Fatal on failure, deliberately."""
    global _SCORER_VERIFIED
    sc = build_scorer()
    probe = GameState("CRANE", VOCAB.n_answers).prompt(1, MAX_GUESSES)
    d = sc.self_test(model, probe)
    print(f"  cache vs naive [{d['dtype']}]: max|delta| = {d['max_abs_dev']:.4f} "
          f"nats (tol {d['atol']}) over a {d['spread']:.1f}-nat spread, "
          f"ranking corr = {d['corr']:.6f}  OK")
    if CONSTRAINED_PRUNE:
        w, nch = sc.verify_against_full(model, probe)
        print(f"  pruned argmax == full argmax ({w}), {nch}/"
              f"{-(-sc.n // CONSTRAINED_CHUNK)} chunks  OK")
    full = sc.score_all(model, probe)
    want = [sc.words[i] for i in torch.argsort(full, descending=True)[:4].tolist()]
    got, ban = [], []
    for _ in range(4):
        w2, _, _, _ = sc.argmax(model, probe, banned=ban or None)
        assert w2 not in ban, f"banned word {w2} returned"
        got.append(w2); ban.append(w2)
    assert got == want, f"banning broke the ranking: {got} != {want}"
    print(f"  sequential banning walks the global ranking {got}  OK")
    _SCORER_VERIFIED = True
    return d

print("constrained decoder ready.")

---
# 8. Evaluation harness

Identical protocol to Phase 5. The only new knob is `ban_repeats`, which is a
decoder setting, not a change to what the model sees.

In [ ]:
import re
from peft import PeftModel

WORD_RE = re.compile(r"^[A-Za-z]{5}$")

def extract_guess(text):
    line = text.strip().split("\n")[0]
    toks = [t.strip(".,:;!?\"'()[]*_-") for t in line.split()]
    toks = [t for t in toks if t]
    if not toks:
        return None, "invalid_format"
    if WORD_RE.match(toks[0]):
        return toks[0].upper(), "ok"
    return None, "invalid_format"

class GameState:
    __slots__ = ("answer", "history", "cands", "guesses", "patterns",
                 "remaining", "statuses", "margins", "done", "solved", "mode")
    def __init__(self, answer, n_answers, mode="constrained"):
        self.answer = answer; self.mode = mode
        self.history = []
        self.cands = np.arange(n_answers, dtype=np.int32)
        self.guesses, self.patterns, self.remaining = [], [], []
        self.statuses, self.margins = [], []
        self.done = self.solved = False
    def prompt(self, turn, max_guesses):
        return render_prompt(
            turn=turn,
            history=[(g.lower(), p) for g, p in self.history],
            constraints=derive_constraints([(g.lower(), p) for g, p in self.history]),
            n_candidates=len(self.cands),   # analysis only
            guesses_remaining=max_guesses - turn + 1,
            max_guesses=max_guesses,
            candidates=None, show_candidate_count=False)

def _apply(g, word, status, turn, max_guesses, margin=None):
    g.statuses.append(status); g.margins.append(margin)
    if status != "ok":
        g.guesses.append(word or "<INVALID>")
        g.patterns.append(None); g.remaining.append(int(len(g.cands)))
    else:
        code_ = feedback_code(word.lower(), g.answer.lower())
        pat = code_to_pattern(code_)
        g.cands = BUNDLE.fb.filter_indices(g.cands, word.lower(), code_)
        g.history.append((word, pat))
        g.guesses.append(word); g.patterns.append(pat)
        g.remaining.append(int(len(g.cands)))
        if code_ == ALL_GREEN:
            g.solved = g.done = True
    if turn == max_guesses and not g.solved:
        g.done = True

PLAY_STATS = {}

@torch.no_grad()
def play_games(model, answers, mode="constrained", ban_repeats=False,
               scorer=None, max_guesses=MAX_GUESSES, batch=EVAL_BATCH):
    model.eval()
    tok = TOKENIZER
    old = tok.padding_side
    tok.padding_side = "left"
    games = [GameState(a, VOCAB.n_answers, mode=mode) for a in answers]
    chunks = []
    t0 = time.perf_counter()
    for turn in range(1, max_guesses + 1):
        active = [g for g in games if not g.done]
        if not active:
            break
        if mode == "constrained":
            for i, g in enumerate(active):
                p = g.prompt(turn, max_guesses)
                banned = list(dict.fromkeys(g.guesses)) if ban_repeats else None
                w, s, nch, margin = scorer.argmax(model, p, banned=banned,
                                                  prune=CONSTRAINED_PRUNE)
                chunks.append(nch)
                _apply(g, w, "ok", turn, max_guesses, margin)
                if (i + 1) % 60 == 0:
                    el = time.perf_counter() - t0
                    print(f"      turn {turn}: {i+1}/{len(active)} games  "
                          f"{el:.0f}s  avg {np.mean(chunks):.1f} chunks",
                          flush=True)
        else:
            for s0 in range(0, len(active), batch):
                chunk = active[s0:s0 + batch]
                enc = tok([g.prompt(turn, max_guesses) for g in chunk],
                          return_tensors="pt", padding=True, truncation=True,
                          max_length=MAX_SEQ_LEN).to(model.device)
                out = model.generate(**enc, max_new_tokens=GEN_MAX_NEW_TOKENS,
                                     do_sample=False, temperature=None,
                                     top_p=None, top_k=None,
                                     pad_token_id=tok.pad_token_id)
                texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                         skip_special_tokens=True)
                for g, txt in zip(chunk, texts):
                    w, st = extract_guess(txt)
                    if w is not None and w not in LEGAL_GUESSES:
                        st = "invalid_word"
                    _apply(g, w, st, turn, max_guesses)
        done = sum(1 for g in games if g.done)
        print(f"  turn {turn}: {done}/{len(games)} finished "
              f"({time.perf_counter()-t0:.0f}s)", flush=True)
    tok.padding_side = old
    PLAY_STATS.clear()
    if chunks:
        PLAY_STATS.update(avg_chunks_per_decision=round(float(np.mean(chunks)), 2),
                          unpruned_chunks=-(-len(LEGAL_GUESSES)//CONSTRAINED_CHUNK))
    return games

def score_games(games, label, max_guesses=MAX_GUESSES):
    n = len(games)
    scores = [len(g.guesses) if g.solved else max_guesses + 1 for g in games]
    solved = [len(g.guesses) for g in games if g.solved]
    dist = {k: sum(1 for g in games if g.solved and len(g.guesses) == k)
            for k in range(1, max_guesses + 1)}
    cum = lambda m: 100 * sum(dist[i] for i in range(1, m + 1)) / n
    st = [s for g in games for s in g.statuses]
    rate = lambda x: 100 * sum(1 for s in st if s == x) / max(len(st), 1)
    rep = sum(1 for g in games if len(g.guesses) != len(set(g.guesses)))
    hv = tv = 0
    for g in games:
        seen = []
        for gu, pat in zip(g.guesses, g.patterns):
            if pat is None:
                continue
            tv += 1
            greens = {}
            for pg, pp in seen:
                for i, (ch, t) in enumerate(zip(pg, pp)):
                    if t == "G":
                        greens[i] = ch
            if any(gu[i] != ch for i, ch in greens.items()):
                hv += 1
            seen.append((gu, pat))
    margins = [m for g in games for m in g.margins if m is not None]
    return {
        "model": label, "mode": games[0].mode, "n_games": n,
        "mean_failures_as_7": round(sum(scores)/n, 4),
        "mean_solved_only": round(sum(solved)/len(solved), 4) if solved else None,
        "median": float(np.median(solved)) if solved else None,
        "max": max(solved) if solved else None,
        "pct_le3": round(cum(3), 2), "pct_le4": round(cum(4), 2),
        "pct_le5": round(cum(5), 2), "pct_le6": round(cum(6), 2),
        "distribution": dist, "failures": n - len(solved),
        "failure_rate_pct": round(100*(n-len(solved))/n, 2),
        "turns_generated": len(st),
        "invalid_format_rate_pct": round(rate("invalid_format"), 2),
        "invalid_word_rate_pct": round(rate("invalid_word"), 2),
        "repeated_guess_game_rate_pct": round(100*rep/n, 2),
        "avg_unique_guesses_per_game": round(
            float(np.mean([len(set(g.guesses)) for g in games])), 3),
        "hard_mode_violation_pct": round(100*hv/max(tv, 1), 2),
        "avg_candidates_after_turn": {
            str(t): round(float(np.mean([g.remaining[t-1] for g in games
                                         if len(g.remaining) >= t])), 2)
            for t in range(1, max_guesses+1)
            if any(len(g.remaining) >= t for g in games)},
        "avg_decision_margin": round(float(np.mean(margins)), 4) if margins else None,
    }

def load_adapter(name):
    path = os.path.join(WORK_DIR, name)
    if not os.path.exists(os.path.join(path, "adapter_config.json")):
        alt = os.path.join(PREV_RUN_DIR or "", name)
        assert os.path.exists(os.path.join(alt, "adapter_config.json")), (
            f"no adapter for {name} in {path} or {alt}. For '{ADAPTER_NAME}' "
            f"run section 6; for 'tree_salet' point PREV_RUN_DIR at the Phase 5 "
            f"adapters dataset.")
        path = alt
    m = PeftModel.from_pretrained(load_base_model(), path)
    m.config.use_cache = True
    return m.eval().cuda()

print("harness ready.  candidate count/list/answer shown: False (never)")

---
# 9. Result containers

Their own cell, before any loop. In Phase 5 these lived at the end of the
evaluation cell and a single interrupt left them undefined, which broke the save
step. `save_state()` is called after **every** evaluation so a session teardown
cannot cost more than one cell.

In [ ]:
EVAL_ROWS = {}          # (adapter, ban) -> metrics
GAMES_ALL = {}          # (adapter, ban) -> [GameState]
BASELINES = []
TERMINAL = {}
TERMINAL_ROWS = []
TERMINAL_SPLIT = {}
OPENERS = {}
DISAGREE = {}
EXAMPLES = {}
TRAIN_TARGET_WORDS = set()

def _key(a, ban):
    return f"{a}__{'ban' if ban else 'noban'}"

def save_state(tag=""):
    """Write everything known so far. Cheap, and called constantly."""
    env = {"python": platform.python_version(), "torch": torch.__version__,
           "transformers": transformers.__version__, "peft": peft.__version__,
           "numpy": np.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
           "fp16": FP16, "seed": SEED,
           "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    payload = {
        "phase": 6, "note": f"{len(VAL_ANSWERS)} HELD-OUT answers.",
        "adapter": ADAPTER_NAME, "model_name": MODEL_NAME,
        "training": TRAIN_CONFIG, "data_stats": DATA_STATS,
        "eval_rows": EVAL_ROWS, "classical_baselines": BASELINES,
        "terminal_probe": TERMINAL, "terminal_seen_vs_unseen": TERMINAL_SPLIT,
        "openers": OPENERS, "disagreement": DISAGREE, "examples": EXAMPLES,
        "phase5_reference": PHASE5, "environment": env,
        "eval_settings": {
            "constrained_legal_words": len(LEGAL_WORDS_SORTED),
            "chunk": CONSTRAINED_CHUNK, "exact_pruning": CONSTRAINED_PRUNE,
            "length_normalise": LENGTH_NORMALISE,
            "candidate_list_shown": False, "candidate_count_shown": False,
            "answer_shown": False,
            "score": "sum log P(token | prompt, prefix) over ' '+WORD tokens "
                     "and EOS; argmax over the legal set."},
    }
    with open(os.path.join(RESULTS_ROOT, "results.json"), "w",
              encoding="utf-8") as fh:
        json.dump(payload, fh, indent=2, default=str)
    if tag:
        print(f"    [saved: {tag}]", flush=True)

save_state("init")
print(f"containers ready; results -> {RESULTS_ROOT}/results.json")

---
# 10. The 2×2

`tree_salet` + `noban` reproduces the Phase 5 measurement (**5.6870**). If it
does not come back within rounding, the environment differs and nothing else in
this notebook should be trusted.

In [ ]:
if RUN_EVALUATION:
    for adapter, ban in EVAL_MATRIX:
        k = _key(adapter, ban)
        if k in EVAL_ROWS:
            print(f"{k}: already done, skipping"); continue
        print("=" * 66)
        print(f"EVALUATING {adapter}  ban_repeats={ban}  on {len(VAL_ANSWERS)} games")
        print("=" * 66)
        try:
            model = load_adapter(adapter)
        except AssertionError as e:
            print(f"  SKIPPED: {e}\n"); continue
        if not _SCORER_VERIFIED:
            verify_scorer(model)
        sc = build_scorer()
        t0 = time.perf_counter()
        gs = play_games(model, VAL_ANSWERS, mode="constrained",
                        ban_repeats=ban, scorer=sc)
        row = score_games(gs, f"{adapter} [{'ban' if ban else 'noban'}]")
        row.update(PLAY_STATS)
        row["adapter"] = adapter
        row["ban_repeats"] = ban
        row["eval_seconds"] = round(time.perf_counter() - t0, 1)
        GAMES_ALL[k] = gs
        EVAL_ROWS[k] = row
        print(f"  mean={row['mean_failures_as_7']:.4f}  "
              f"fail={row['failure_rate_pct']:.1f}%  "
              f"solved={100-row['failure_rate_pct']:.1f}%  "
              f"repeat={row['repeated_guess_game_rate_pct']:.1f}%  "
              f"invalid={row['invalid_word_rate_pct']:.1f}%  "
              f"({row['eval_seconds']:.0f}s)")
        if adapter == "tree_salet" and not ban:
            d = row["mean_failures_as_7"] - PHASE5["tree_salet_constrained_noban"]
            print(f"  PHASE 5 REPRODUCTION CHECK: {row['mean_failures_as_7']:.4f} "
                  f"vs 5.6870 (delta {d:+.4f})"
                  f"{'  OK' if abs(d) < 0.05 else '  <-- MISMATCH, investigate'}")
        print()
        save_state(k)
        del model; torch.cuda.empty_cache()
else:
    print("evaluation skipped")

---
# 11. Classical baselines on the identical 246 answers

In [ ]:
if RUN_BASELINES and not BASELINES:
    ensure_code_on_path()
    from tree_search import TreeSearchConfig, TreeSearchSolver
    cfgc = SolverConfig(max_guesses=MAX_GUESSES, seed=SEED, guess_pool="full")
    lower = [a.lower() for a in VAL_ANSWERS]

    def summarize(label, games, n=len(VAL_ANSWERS)):
        scores = [g.score for g in games]
        solved = [g.n_guesses for g in games if g.solved]
        dist = {k: sum(1 for g in games if g.solved and g.n_guesses == k)
                for k in range(1, 7)}
        cum = lambda m: 100*sum(dist[i] for i in range(1, m+1))/n
        return {"model": label, "n_games": n,
                "mean_failures_as_7": round(sum(scores)/n, 4),
                "median": float(np.median(solved)) if solved else None,
                "max": max(solved) if solved else None,
                "pct_le3": round(cum(3), 2), "pct_le4": round(cum(4), 2),
                "pct_le5": round(cum(5), 2),
                "failure_rate_pct": round(100*(n-len(solved))/n, 2),
                "classical": True}

    for lbl in ["random", "frequency", "entropy"]:
        sv = make_solver(lbl, BUNDLE.fb, cfgc, BUNDLE.model); sv.reset()
        op = sv.opening_guess() if sv.deterministic else None
        BASELINES.append(summarize(lbl, [play_game(sv, a, first_guess=op)
                                         for a in lower]))
        print(f"  {lbl:<12} {BASELINES[-1]['mean_failures_as_7']:.4f}")
    tc = TreeSearchConfig(depth=6, top_k=100, endgame_top_k=60,
                          endgame_threshold=10, opening_guess="salet",
                          max_guesses=MAX_GUESSES)
    sv = TreeSearchSolver(BUNDLE.fb, cfgc, tc)
    BASELINES.append(summarize("tree_salet", [play_game(sv, a, first_guess="salet")
                                              for a in lower]))
    print(f"  {'tree_salet':<12} {BASELINES[-1]['mean_failures_as_7']:.4f}")
    save_state("baselines")
else:
    print("baselines skipped or already computed")

---
# 12. Terminal-state probe

The decisive measurement. The classical solver — not the model — is rolled to
states with exactly `k` candidates. At `k=1` the constraints determine the
answer; nothing remains but naming it.

**Read the seen-vs-unseen split first.** If `tree_salet_endgame` improves on
seen words but not unseen, it memorised harder rather than learning the
procedure, and the intervention failed even if the mean improved.

In [ ]:
def terminal_states(answers, ks, solver_label="entropy"):
    cfgc = SolverConfig(max_guesses=MAX_GUESSES, seed=SEED, guess_pool="full")
    sv = make_solver(solver_label, BUNDLE.fb, cfgc, BUNDLE.model); sv.reset()
    opener = sv.opening_guess()
    out = {k: [] for k in ks}
    want = set(ks)
    for ans in answers:
        low = ans.lower()
        cands = np.arange(VOCAB.n_answers, dtype=np.int32)
        hist, seen = [], set()
        for turn in range(1, MAX_GUESSES + 1):
            n = int(len(cands))
            if n in want and n not in seen:
                seen.add(n)
                out[n].append({
                    "answer": ans, "k": n, "turn": turn,
                    "candidates": [VOCAB.answers[i].upper() for i in cands],
                    "prompt": render_prompt(
                        turn=turn, history=list(hist),
                        constraints=derive_constraints(hist), n_candidates=n,
                        guesses_remaining=MAX_GUESSES - turn + 1,
                        max_guesses=MAX_GUESSES, candidates=None,
                        show_candidate_count=False)})
            if n <= 1 or not want - seen:
                break
            g = opener if turn == 1 else sv.choose(cands, turn)
            c = feedback_code(g, low)
            hist.append((g, code_to_pattern(c)))
            cands = BUNDLE.fb.filter_indices(cands, g, c)
            if c == ALL_GREEN:
                break
    return out

@torch.no_grad()
def probe_terminal(model, states, label, caps):
    sc = build_scorer()
    rows, per_k = [], {}
    plan = {k: (states[k] if caps.get(k) is None else states[k][:caps[k]])
            for k in sorted(states)}
    total = sum(len(v) for v in plan.values())
    print(f"  scoring {total} states "
          f"({', '.join(f'k={k}:{len(v)}' for k, v in plan.items())})")
    t0, done = time.perf_counter(), 0
    for k, use in plan.items():
        hit1 = hitc = 0
        ranks, ranks_in = [], []
        for st in use:
            scores = sc.score_all(model, st["prompt"])
            top1 = sc.words[int(scores.argmax().item())]
            r = sc.rank_of(scores, [st["answer"]]).get(st["answer"])
            cs = sorted(((sc.words[sc.index[c]], scores[sc.index[c]].item())
                         for c in st["candidates"] if c in sc.index),
                        key=lambda x: -x[1])
            r_in = next((i+1 for i, (w, _) in enumerate(cs)
                         if w == st["answer"]), None)
            hit1 += int(top1 == st["answer"])
            hitc += int(top1 in set(st["candidates"]))
            if r: ranks.append(r)
            if r_in: ranks_in.append(r_in)
            rows.append({"model": label, "k": k, "answer": st["answer"],
                         "top1": top1, "correct": top1 == st["answer"],
                         "top1_is_candidate": top1 in set(st["candidates"]),
                         "rank_of_answer": r, "rank_within_candidates": r_in,
                         "answer_in_train_targets":
                             st["answer"] in TRAIN_TARGET_WORDS})
            done += 1
            if PROBE_LOG_EVERY and done % PROBE_LOG_EVERY == 0:
                el = time.perf_counter() - t0
                print(f"    {done}/{total}  {el:.0f}s elapsed, "
                      f"~{el/done*(total-done):.0f}s left", flush=True)
        n = max(len(use), 1)
        per_k[k] = {"n_states": len(use),
                    "top1_accuracy_pct": round(100*hit1/n, 2),
                    "top1_is_candidate_pct": round(100*hitc/n, 2),
                    "median_rank_of_answer": float(np.median(ranks)) if ranks else None,
                    "mean_rank_within_candidates": round(float(np.mean(ranks_in)), 3)
                        if ranks_in else None,
                    "chance_top1_pct": round(100/sc.n, 4)}
    return per_k, rows

if RUN_TERMINAL_PROBE:
    if not TRAIN_TARGET_WORDS:
        for f in ("data/tree_salet/train.jsonl",
                  "data/tree_salet_endgame/train.jsonl"):
            p = os.path.join(SFT_DIR, f)
            if os.path.exists(p):
                for l in open(p, encoding="utf-8"):
                    TRAIN_TARGET_WORDS.add(json.loads(l)["completion"].upper())
    print(f"distinct training targets: {len(TRAIN_TARGET_WORDS)}")

    if "STATES" not in globals():
        STATES = terminal_states(VAL_ANSWERS, sorted(TERMINAL_MAX_STATES))
    for k in sorted(STATES):
        print(f"  k={k}: {len(STATES[k])} states available")

    for adapter in TERMINAL_MODELS:
        if adapter in TERMINAL:
            continue
        print(f"\n--- terminal probe: {adapter} ---", flush=True)
        try:
            model = load_adapter(adapter)
        except AssertionError as e:
            print(f"  SKIPPED: {e}"); continue
        if not _SCORER_VERIFIED:
            verify_scorer(model)
        t0 = time.perf_counter()
        per_k, rows = probe_terminal(model, STATES, adapter, TERMINAL_MAX_STATES)
        TERMINAL[adapter] = per_k
        TERMINAL_ROWS.extend(rows)
        for k, r in sorted(per_k.items()):
            print(f"  k={k}  n={r['n_states']:>3}  "
                  f"top1={r['top1_accuracy_pct']:>6.2f}%  "
                  f"in_candidates={r['top1_is_candidate_pct']:>6.2f}%  "
                  f"median_rank={r['median_rank_of_answer']}")
        print(f"  ({time.perf_counter()-t0:.0f}s)")
        save_state(f"terminal:{adapter}")
        del model; torch.cuda.empty_cache()

    print("\n" + "=" * 70)
    print("k=1 TOP-1, SPLIT BY WHETHER THE ANSWER WAS EVER A TRAINING TARGET")
    print("=" * 70)
    hdr = f"{'model':<24}{'seen n':>8}{'seen':>9}{'unseen n':>10}{'unseen':>9}"
    print(hdr); print("-" * len(hdr))
    for label in sorted({r["model"] for r in TERMINAL_ROWS}):
        sub = [r for r in TERMINAL_ROWS if r["model"] == label and r["k"] == 1]
        seen = [r for r in sub if r["answer_in_train_targets"]]
        uns = [r for r in sub if not r["answer_in_train_targets"]]
        f = lambda xs: round(100*sum(x["correct"] for x in xs)/len(xs), 2) if xs else None
        TERMINAL_SPLIT[label] = {"seen_n": len(seen), "seen_acc_pct": f(seen),
                                 "unseen_n": len(uns), "unseen_acc_pct": f(uns)}
        print(f"{label:<24}{len(seen):>8}{str(f(seen)):>9}"
              f"{len(uns):>10}{str(f(uns)):>9}")
    print(f"\nPhase 5 tree_salet was: seen {PHASE5['k1_seen']}%  "
          f"unseen {PHASE5['k1_unseen']}%  (overall {PHASE5['k1_top1']}%)")
    save_state("terminal-split")
else:
    print("terminal probe skipped")

---
# 13. Comparison and verdict

In [ ]:
import csv

print("=" * 96)
print(f"PHASE 6 vs PHASE 5  -  {len(VAL_ANSWERS)} held-out answers")
print("=" * 96)
hdr = (f"{'model':<34}{'mean':>9}{'fail%':>8}{'solved%':>9}{'repeat%':>9}"
       f"{'<=3':>7}{'<=4':>7}")
print(hdr); print("-" * len(hdr))
for b in BASELINES:
    print(f"{'classical ' + b['model']:<34}{b['mean_failures_as_7']:>9.4f}"
          f"{b['failure_rate_pct']:>8.1f}{100-b['failure_rate_pct']:>9.1f}"
          f"{0.0:>9.1f}{b['pct_le3']:>7.1f}{b['pct_le4']:>7.1f}")
print(f"{'PHASE 5 tree_salet [noban]':<34}"
      f"{PHASE5['tree_salet_constrained_noban']:>9.4f}"
      f"{PHASE5['tree_salet_constrained_noban_fail']:>8.1f}"
      f"{100-PHASE5['tree_salet_constrained_noban_fail']:>9.1f}"
      f"{30.5:>9.1f}{17.1:>7.1f}{30.1:>7.1f}")
for k, r in EVAL_ROWS.items():
    print(f"{r['model']:<34}{r['mean_failures_as_7']:>9.4f}"
          f"{r['failure_rate_pct']:>8.1f}{100-r['failure_rate_pct']:>9.1f}"
          f"{r['repeated_guess_game_rate_pct']:>9.1f}"
          f"{r['pct_le3']:>7.1f}{r['pct_le4']:>7.1f}")

# ---- the two effects, separated -------------------------------------------
def get(a, ban):
    r = EVAL_ROWS.get(_key(a, ban))
    return r["mean_failures_as_7"] if r else None

new_nb, new_b = get(ADAPTER_NAME, False), get(ADAPTER_NAME, True)
old_nb, old_b = get("tree_salet", False), get("tree_salet", True)

print("\n" + "=" * 70); print("EFFECT DECOMPOSITION"); print("=" * 70)
if new_nb is not None and old_nb is not None:
    print(f"  endgame data alone (noban): {old_nb:.4f} -> {new_nb:.4f}  "
          f"({new_nb-old_nb:+.4f})")
if old_b is not None and old_nb is not None:
    print(f"  repeat banning alone (old): {old_nb:.4f} -> {old_b:.4f}  "
          f"({old_b-old_nb:+.4f})")
if new_b is not None and old_nb is not None:
    print(f"  both combined             : {old_nb:.4f} -> {new_b:.4f}  "
          f"({new_b-old_nb:+.4f})")

# ---- verdict ---------------------------------------------------------------
def verdict():
    ev = []
    best = min([v for v in (new_nb, new_b) if v is not None], default=None)
    if best is None:
        return "INCONCLUSIVE", ["the new adapter was not evaluated"]
    rnd = next((b["mean_failures_as_7"] for b in BASELINES
                if b["model"] == "random"), PHASE5["classical_random"])
    ent = next((b["mean_failures_as_7"] for b in BASELINES
                if b["model"] == "entropy"), PHASE5["classical_entropy"])
    ev.append(f"best Phase 6 = {best:.4f}   Phase 5 = "
              f"{PHASE5['tree_salet_constrained_noban']:.4f}")
    ev.append(f"classical random = {rnd:.4f}, entropy = {ent:.4f}")
    sp = TERMINAL_SPLIT.get(ADAPTER_NAME)
    if sp:
        ev.append(f"k=1 seen {sp['seen_acc_pct']}% (n={sp['seen_n']}) vs "
                  f"unseen {sp['unseen_acc_pct']}% (n={sp['unseen_n']})  "
                  f"[Phase 5: {PHASE5['k1_seen']}% vs {PHASE5['k1_unseen']}%]")
    t = TERMINAL.get(ADAPTER_NAME, {}).get(1)
    if t:
        ev.append(f"k=1 top-1 {t['top1_accuracy_pct']}% "
                  f"[Phase 5: {PHASE5['k1_top1']}%], median rank "
                  f"{t['median_rank_of_answer']} "
                  f"[Phase 5: {PHASE5['k1_median_rank']}]")

    gen = None
    if sp and sp["unseen_acc_pct"] is not None:
        gen = sp["unseen_acc_pct"] - PHASE5["k1_unseen"]

    if best <= ent + 0.35:
        return "SOLVED", ev + ["-> reaches the classical expert"]
    if best < rnd:
        return "MAJOR", ev + ["-> now beats random elimination; endgame "
                              "coverage was the binding constraint"]
    if gen is not None and gen >= 10:
        return "PARTIAL-GENERALISING", ev + [
            "-> still short of random, but unseen-word accuracy moved "
            f"{gen:+.1f} pts: the procedure is transferring. More/denser "
            "paths per word is the obvious next lever."]
    if gen is not None and gen < 3:
        return "MEMORISED", ev + [
            "-> unseen-word accuracy barely moved. The extra paths bought "
            "memorisation, not a procedure. More of the same data will not "
            "help; the task framing needs to change (rank the candidate set "
            "rather than emit from memory)."]
    return "INCONCLUSIVE", ev + ["-> no clean reading; inspect the tables"]

V, EVIDENCE = verdict()
print("\n" + "=" * 70); print(f"VERDICT: {V}"); print("=" * 70)
for e in EVIDENCE:
    print("  " + e)

# ---- write artifacts -------------------------------------------------------
FIELDS = ["model", "adapter", "ban_repeats", "n_games", "mean_failures_as_7",
          "failure_rate_pct", "repeated_guess_game_rate_pct",
          "invalid_word_rate_pct", "pct_le3", "pct_le4", "pct_le5",
          "median", "max", "avg_chunks_per_decision", "eval_seconds"]
with open(os.path.join(RESULTS_ROOT, "comparison.csv"), "w", newline="",
          encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=FIELDS, extrasaction="ignore")
    w.writeheader()
    for r in EVAL_ROWS.values():
        w.writerow(r)
if TERMINAL_ROWS:
    with open(os.path.join(RESULTS_ROOT, "terminal_probe.csv"), "w",
              newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(TERMINAL_ROWS[0].keys()))
        w.writeheader()
        for r in TERMINAL_ROWS:
            w.writerow(r)
json.dump({"verdict": V, "evidence": EVIDENCE},
          open(os.path.join(RESULTS_ROOT, "verdict.json"), "w",
               encoding="utf-8"), indent=2)
save_state("final")

base = RESULTS_ZIP[:-4] if RESULTS_ZIP.endswith(".zip") else RESULTS_ZIP
shutil.make_archive(base, "zip", RESULTS_ROOT)
print(f"\nRESULTS ZIP: {base}.zip "
      f"({os.path.getsize(base + '.zip')/2**20:.2f} MiB)  -- no weights inside")
print(f"adapter kept at {WORK_DIR}/{ADAPTER_NAME} "
      f"(snapshot it as a Dataset to reuse next session)")